In [1]:
# Run notebook from repo root: uv run jupyter notebook scripts/test_rrweb_video.ipynb
# Or ensure project root is on path when kernel starts from scripts/
import os
import sys
import time
import json

_cwd = os.getcwd()
_repo_root = os.path.dirname(_cwd) if os.path.basename(_cwd) == "scripts" else _cwd
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)
os.chdir(_repo_root)

from graphs.video_recording_analyzer_graph import VideoRecordingAnalyzerGraph
from utils.recording import (
    get_recording_duration,
    is_ffmpeg_available,
    ffmpeg_chunk_video,
    chunk_video as opencv_chunk_video,
)
from common.services.logger import logger
from settings import settings

logger.info("Notebook setup", repo_root=_repo_root, cwd=os.getcwd())


2026-01-31 21:50:41,880 INFO Notebook setup repo_root: /Users/abesolyman/Desktop/Work/perceptr/perceptr,cwd: /Users/abesolyman/Desktop/Work/perceptr/perceptr


In [3]:
# Config: video path, chunk size, output dir for chunks
VIDEO_PATH = os.path.join(_repo_root, "scripts", "assets", "t.mp4")
CHUNK_SIZE_SECONDS = getattr(settings, "RECORDING_INTERVAL_DURATION", 30)
CHUNKS_OUTPUT_DIR = os.path.join(_repo_root, "tmp_session", "video_chunks")
FILE_TYPE = "video/mp4"
TEST_ORG_ID = "test-org"
TEST_RECORDING_ID = "test-recording-video"

os.makedirs(CHUNKS_OUTPUT_DIR, exist_ok=True)
assert os.path.isfile(VIDEO_PATH), f"Video not found: {VIDEO_PATH}"

logger.info(
    "Video test config",
    video_path=VIDEO_PATH,
    chunk_size_seconds=CHUNK_SIZE_SECONDS,
    chunks_output_dir=CHUNKS_OUTPUT_DIR,
    file_type=FILE_TYPE,
)
print(f"Video: {VIDEO_PATH}")
print(f"Chunk size: {CHUNK_SIZE_SECONDS}s | Chunks dir: {CHUNKS_OUTPUT_DIR}")


2026-01-31 21:51:10,625 INFO Video test config video_path: /Users/abesolyman/Desktop/Work/perceptr/perceptr/scripts/assets/t.mp4,chunk_size_seconds: 30,chunks_output_dir: /Users/abesolyman/Desktop/Work/perceptr/perceptr/tmp_session/video_chunks,file_type: video/mp4


Video: /Users/abesolyman/Desktop/Work/perceptr/perceptr/scripts/assets/t.mp4
Chunk size: 30s | Chunks dir: /Users/abesolyman/Desktop/Work/perceptr/perceptr/tmp_session/video_chunks


In [4]:
# Get video duration and chunk the video
use_ffmpeg = is_ffmpeg_available()
logger.info("Chunking method", use_ffmpeg=use_ffmpeg)

if use_ffmpeg:
    from utils.recording import ffmpeg_get_video_duration
    duration_s = ffmpeg_get_video_duration(VIDEO_PATH)
    logger.info("Video duration (FFmpeg)", duration_seconds=duration_s, video_path=VIDEO_PATH)
else:
    duration_s = get_recording_duration(VIDEO_PATH)
    logger.info("Video duration (OpenCV)", duration_seconds=duration_s, video_path=VIDEO_PATH)

_chunk_start = time.perf_counter()
if use_ffmpeg:
    chunk_info = ffmpeg_chunk_video(VIDEO_PATH, CHUNK_SIZE_SECONDS, output_dir=CHUNKS_OUTPUT_DIR)
else:
    chunk_info = opencv_chunk_video(VIDEO_PATH, CHUNK_SIZE_SECONDS, output_dir=CHUNKS_OUTPUT_DIR)
_chunk_elapsed = time.perf_counter() - _chunk_start

logger.info(
    "Chunking complete",
    duration_seconds=duration_s,
    chunk_count=len(chunk_info),
    chunk_size_seconds=CHUNK_SIZE_SECONDS,
    chunking_time_seconds=round(_chunk_elapsed, 2),
)
for i, (path, start_s, dur) in enumerate(chunk_info):
    logger.info("Chunk", index=i + 1, path=path, start_seconds=start_s, duration_seconds=dur)
print(f"Duration: {duration_s:.1f}s | Chunks: {len(chunk_info)} | Chunking time: {_chunk_elapsed:.2f}s")


2026-01-31 21:51:12,805 INFO Chunking method use_ffmpeg: False
2026-01-31 21:51:12,826 INFO Video duration (OpenCV) duration_seconds: 116.88,video_path: /Users/abesolyman/Desktop/Work/perceptr/perceptr/scripts/assets/t.mp4


Created chunk 1/4 from 0s to 30s
Created chunk 2/4 from 30s to 60s
Created chunk 3/4 from 60s to 90s


2026-01-31 21:52:03,084 INFO Chunking complete duration_seconds: 116.88,chunk_count: 4,chunk_size_seconds: 30,chunking_time_seconds: 50.26
2026-01-31 21:52:03,086 INFO Chunk index: 1,path: /Users/abesolyman/Desktop/Work/perceptr/perceptr/tmp_session/video_chunks/t_chunk_0.mp4,start_seconds: 0,duration_seconds: 30
2026-01-31 21:52:03,086 INFO Chunk index: 2,path: /Users/abesolyman/Desktop/Work/perceptr/perceptr/tmp_session/video_chunks/t_chunk_1.mp4,start_seconds: 30,duration_seconds: 30
2026-01-31 21:52:03,086 INFO Chunk index: 3,path: /Users/abesolyman/Desktop/Work/perceptr/perceptr/tmp_session/video_chunks/t_chunk_2.mp4,start_seconds: 60,duration_seconds: 30
2026-01-31 21:52:03,087 INFO Chunk index: 4,path: /Users/abesolyman/Desktop/Work/perceptr/perceptr/tmp_session/video_chunks/t_chunk_3.mp4,start_seconds: 90,duration_seconds: 26.879999999999995


Created chunk 4/4 from 90s to 116.88s
Duration: 116.9s | Chunks: 4 | Chunking time: 50.26s


In [ ]:
# Run VideoRecordingAnalyzerGraph on each chunk — time, cost note, output logged
video_graph = VideoRecordingAnalyzerGraph()
all_intervals = []
chunk_times = []
chunk_outputs = []
raw_events = []

logger.info("Starting chunk-wise analysis", chunk_count=len(chunk_info), recording_id=TEST_RECORDING_ID)
total_start = time.perf_counter()

for i, (chunk_path, start_seconds, duration) in enumerate(chunk_info):
    chunk_id = f"{TEST_RECORDING_ID}_chunk_{start_seconds}"
    t0 = time.perf_counter()
    try:
        resp = video_graph.analyze_recording(
            org_id=TEST_ORG_ID,
            recording_id=chunk_id,
            recording_path=chunk_path,
            file_type=FILE_TYPE,
        )
        elapsed = time.perf_counter() - t0
        chunk_times.append(elapsed)
        raw_events.append({"chunk_id": i, "response": resp})
        ra = resp.get("recording_analysis")
        if ra:
            all_intervals.extend(ra.intervals)
            chunk_outputs.append({
                "chunk_index": i + 1,
                "start_seconds": start_seconds,
                "duration_seconds": duration,
                "time_seconds": round(elapsed, 2),
                "title": ra.title,
                "interval_count": len(ra.intervals),
                "summary": (ra.summary or ""),
            })
        logger.info(
            "Chunk analysis done",
            chunk_index=i + 1,
            chunk_id=chunk_id,
            time_seconds=round(elapsed, 2),
            interval_count=len(ra.intervals) if ra else 0,
            cost_note="See Langfuse dashboard for token/cost for this run",
            output_title=ra.title if ra else None,
        )
        print(f"Chunk {i+1}/{len(chunk_info)}: {elapsed:.2f}s | intervals: {len(ra.intervals) if ra else 0} | cost: see Langfuse")
    except Exception as e:
        elapsed = time.perf_counter() - t0
        logger.error("Chunk analysis failed", chunk_index=i + 1, chunk_id=chunk_id, time_seconds=round(elapsed, 2), exc_info=e)
        raise

total_elapsed = time.perf_counter() - total_start
logger.info(
    "All chunks analyzed",
    total_time_seconds=round(total_elapsed, 2),
    total_intervals=len(all_intervals),
    cost_note="Aggregate cost: check Langfuse for trace/session",
    chunk_times_seconds=[round(t, 2) for t in chunk_times],
)
print(f"\nTotal analysis time: {total_elapsed:.2f}s | Total intervals: {len(all_intervals)}")


2026-01-31 21:52:07,074 INFO Starting chunk-wise analysis chunk_count: 4,recording_id: test-recording-video
2026-01-31 21:52:07,126 INFO Uploading video file: /Users/abesolyman/Desktop/Work/perceptr/perceptr/tmp_session/video_chunks/t_chunk_0.mp4 using File API... 
2026-01-31 21:52:15,703 INFO File uploaded successfully file_uri: https://generativelanguage.googleapis.com/v1beta/files/y3b5k9u5o6ge,file_name: files/y3b5k9u5o6ge
2026-01-31 21:52:15,707 INFO Waiting for video processing... file_uri: https://generativelanguage.googleapis.com/v1beta/files/y3b5k9u5o6ge,file_name: files/y3b5k9u5o6ge
2026-01-31 21:52:21,103 INFO Video file ready file_uri: https://generativelanguage.googleapis.com/v1beta/files/y3b5k9u5o6ge,file_name: files/y3b5k9u5o6ge
2026-01-31 21:52:57,713 INFO Deleting uploaded file file_uri: https://generativelanguage.googleapis.com/v1beta/files/y3b5k9u5o6ge
2026-01-31 21:52:59,584 INFO Chunk analysis done chunk_index: 1,chunk_id: test-recording-video_chunk_0,time_seconds: 

Chunk 1/4: 52.51s | intervals: 5 | cost: see Langfuse

Total analysis time: 52.51s | Total intervals: 5


In [6]:
# Summary: cost, time, output — all visible in logs
summary = {
    "cost": "See Langfuse dashboard for token usage and cost (session_id / recording_id per chunk)",
    "time": {
        "chunking_seconds": round(_chunk_elapsed, 2),
        "analysis_total_seconds": round(total_elapsed, 2),
        "analysis_per_chunk_seconds": [round(t, 2) for t in chunk_times],
        "wall_clock_total_seconds": round(_chunk_elapsed + total_elapsed, 2),
    },
    "output": {
        "total_intervals": len(all_intervals),
        "chunk_outputs": chunk_outputs,
        "interval_previews": [
            {"start": inv.start_time, "end": inv.end_time, "short_title": inv.short_title}
            for inv in all_intervals
        ],
    },
}
logger.info(
    "Video test summary",
    cost_note=summary["cost"],
    chunking_seconds=summary["time"]["chunking_seconds"],
    analysis_total_seconds=summary["time"]["analysis_total_seconds"],
    wall_clock_total_seconds=summary["time"]["wall_clock_total_seconds"],
    total_intervals=summary["output"]["total_intervals"],
)
print(json.dumps(summary, indent=2, default=str))


2026-01-31 21:53:52,150 INFO Video test summary cost_note: See Langfuse dashboard for token usage and cost (session_id / recording_id per chunk),chunking_seconds: 50.26,analysis_total_seconds: 52.51,wall_clock_total_seconds: 102.77,total_intervals: 5


{
  "cost": "See Langfuse dashboard for token usage and cost (session_id / recording_id per chunk)",
  "time": {
    "chunking_seconds": 50.26,
    "analysis_total_seconds": 52.51,
    "analysis_per_chunk_seconds": [
      52.51
    ],
    "wall_clock_total_seconds": 102.77
  },
  "output": {
    "total_intervals": 5,
    "chunk_outputs": [
      {
        "chunk_index": 1,
        "start_seconds": 0,
        "duration_seconds": 30,
        "time_seconds": 52.51,
        "title": "Landing Page Registration and Navigation Analysis",
        "interval_count": 5,
        "summary": "The user session focused on reviewing the product offering, signing up via the lead generation form, and attempting to explore further site content. The user successfully completed the 'Reserve Your Spot' form, which provided clear feedback. However, two significant issues were identified: a severe performance lag (5 seconds) during the initial page load, and a broken navigation link in the footer ('Roadmap') 

In [ ]:
# output raw events
for e in raw_events:
    print(e)


{'chunk_id': 0, 'response': {'recording_path': '/Users/abesolyman/Desktop/Work/perceptr/perceptr/tmp_session/video_chunks/t_chunk_0.mp4', 'recording_analysis': RecordingAnalysis(intervals=[TimestampInterval(start_time='00:00', end_time='00:13', description='The session begins with a blank white screen for several seconds, indicating a slow initial page load. At 00:05, the pricing page content appears. The user briefly observes the pricing plans, hovering over the "Get Started" button for the "Starter" plan. They then scroll down the page, passing various sections of the homepage. At 00:13, the user clicks the "Reserve Your Spot" call-to-action button.', findings=[Finding(description='The initial page load displays a blank white screen for approximately 4 seconds (00:01 to 00:05) before the content renders, which is a noticeable delay.', category='PERFORMANCE_ISSUE')], short_title='Initial Load and CTA Click', timestamp_descriptions=[TimestampDescription(description='The screen is blank

In [ ]:
# Optional: cleanup chunk files (same as production service)
for _path, _start, _dur in chunk_info:
    try:
        os.remove(_path)
        logger.info("Removed chunk file", path=_path)
    except Exception as e:
        logger.warning("Failed to remove chunk file", path=_path, exc_info=str(e))
print("Chunk files cleaned up.")
